# 🚀 Deploy LiveTalking AI Service - Google Colab (FREE)

**Mục đích:** Chạy avatar AI phỏng vấn với GPU T4 miễn phí
**Thời gian:** ~20 phút setup ban đầu, session ~12h
**Chi phí:** $0/month - Hoàn toàn miễn phí!

---
**Cách sử dụng:** Chạy từng cell theo thứ tự bằng cách click ▶️ hoặc F5

In [ ]:
# Bước 1: Cài đặt Docker và NVIDIA toolkit (~1 phút)
!apt-get update && apt-get install -y docker.io curl wget dnsutils
!curl -fsSL https://get.docker.com | sh

# Configure Docker for GPU support
import json
with open('/etc/docker/daemon.json', 'w') as f:
    json.dump({
        "runtimes": {
            "nvidia": {"path": "nvidia-container-runtime", "arguments": []}
        }
    }, f)

!systemctl restart docker || service docker restart || true

# Verify GPU access
!nvidia-smi
print('✅ Docker + GPU đã sẵn sàng!')

In [ ]:
# Bước 2: Clone repository BE của bạn (~30 giây)
!git clone https://github.com/PNV-do-an/AI-Interview-Platform-BE.git
%cd AI-Interview-Platform-BE

# Kiểm tra cấu trúc
!ls -la | head -20
!cat requirements-ai.txt

print("✅ Repository đã cloned thành công!")

In [ ]:
# Bước 3: Build Docker image (~15-20 phút) ⏳
print('⏳ Đang build Docker image... Vui lòng đợi (không đóng tab!)')

!docker build -t hiremind-ai-service .

# Verify build thành công
!docker images hiremind-ai-service --format "{{.Repository}}:{{.Tag}}\t{{.Size}}"

print('✅ Docker image đã build xong!')

In [ ]:
# Bước 4: Run container với GPU (~1 phút)
!docker run -d \
  --name hiremind-ai \
  --gpus all \
  -p 8010:8010 \
  -p 8020:8020 \
  -e AI_AVATAR_SERVER=http://livetalking:8010 \
  -e AI_ALLOWED_ORIGIN=* \
  -e AI_HOST=0.0.0.0 \
  -e AI_PORT=8020 \
  -e AI_MAX_TURNS=40 \
  -e AI_RUBRIC_VERSION=2026-10-v1 \
  hiremind-ai-service

# Kiểm tra logs
!docker logs hiremind-ai --tail 30

print('🎭 Container đang chạy!')

In [ ]:
# Bước 5: Forward ports qua ngrok (lấy URL public) (~2 phút) 🎯
!pip install pyngrok -q

from pyngrok import ngrok
import time

print('⏳ Đang tạo tunnel...')
time.sleep(2)

# Create tunnel to FastAPI service (port 8020)
public_url = ngrok.connect(8020)
print(f"\n✅ AI Service URL:\n   {public_url}\n")

# Also expose avatar interface
ngrok_avatar = ngrok.connect(8010)
print(f"🎭 Avatar Interface (test):\n   {ngrok_avatar.public_url}\n")

print('=' * 70)
print('💡 LƯU QUAN TRỌNG:')
print('=' * 70)
print(f'Coppy URL sau vào Vercel Dashboard:')
print(f'REACT_APP_AI_SERVICE_URL = {public_url}')
print('=' * 70)

In [ ]:
# Bước 6: Verify service health (~1 phút)
import requests
import time

def check_health():
    try:
        resp = requests.get('http://localhost:8020/health', timeout=5)
        return resp.json()
    except Exception as e:
        return {'error': str(e)}

# Wait & retry up to 30 times
for i in range(30):
    health = check_health()
    if 'status' in health and health['status'] == 'ok':
        print(f"✅ Service khỏe mạnh sau {i*2} giây!")
        break
    else:
        print(f"⏳ Đợi service khởi động... ({i+1}/30)")
        time.sleep(2)

import json
print(json.dumps(health, indent=2))

In [ ]:
# Bước 7: Get final deployment info
import os

# Get container info
container_id = os.popen('docker ps --filter name=hiremind-ai --format {{.ID}}').read().strip()

# Get GPU usage
gpu_info = os.popen('nvidia-smi --query-gpu=index,name,utilization.gpu,memory.total,memory.used --format=csv,noheader,nounits').read()

# Print summary
print('=' * 70)
print('🎉 DEPLOYMENT HOÀN TẤT!')
print('=' * 70)

print('\n📋 THONG TIN CẦN LƯU:')
print('-' * 70)
print(f'✅ AI Service URL (cho FE):')
print(f'   {public_url}')
print(f'\n🎭 Avatar Test Interface:')
print(f'   {ngrok_avatar.public_url}')
print(f'\n📦 Container ID:')
print(f'   {container_id}')
print(f'\n🖥️ GPU Usage:')
print(gpu_info.replace(',', ' | '))

print('\n🔧 BƯỚC TIẾP THEO:')
print('-' * 70)
print('1. Copy AI Service URL ở trên')
print('2. Vào Vercel Dashboard: https://vercel.com/dashboard')
print('3. Chọn project: AI-Interview-Platform-FE')
print('4. Settings → Environment Variables')
print('5. Add variable:')
print('   Name: REACT_APP_AI_SERVICE_URL')
print(f'   Value: {public_url}')
print('6. Deploy lại app')
print('7. Test tại: https://aiinterview-vn.vercel.app/mock-interviews')
print('8. Chọn mode: "LiveTalking Avatar"')

print('\n💰 CHI PHÍ:')
print('-' * 70)
print('✅ Google Colab Free Tier: $0/month')
print('✅ Ngrok Free: $0/month')
print('✅ Vercel Free: $0/month')
print('= TOTAL: $0 ✨ Hoàn toàn miễn phí!')

# Save for easy copy-paste
with open('/tmp/deployment_info.txt', 'w') as f:
    f.write("=" * 70 + "\n")
    f.write("DEPLOY LIVE TALKING ON COLAB - SUCCESSFUL!\n\n")
    f.write("AI SERVICE URL (copy this to Vercel):\n")
    f.write(f"{public_url}\n\n")
    f.write("AVATAR INTERFACE (test):\n")
    f.write(f"{ngrok_avatar.public_url}\n\n")
    f.write("CONTAINER ID:\n")
    f.write(f"{container_id}\n\n")
    f.write("GPU USAGE:\n")
    f.write(f"{gpu_info.replace(chr(44), ' | ').strip()}\n")
    f.write("\n" + "=" * 70 + "\n")

print('\n📁 Đã lưu thông tin tại: /tmp/deployment_info.txt')
print('✅ Sẵn sàng để deploy! 💪')

In [ ]:
# Bonus: Real-time monitoring
print("=" * 60)
print("📊 REAL-TIME MONITORING")
print("=" * 60)
print(f"Container Status:")
!docker ps --filter name=hiremind-ai --format "table {{.Status}}\t{{.Names}}\t{{.Ports}}"

print(f"\nGPU Usage:")
!nvidia-smi

print(f"\nLogs (last 10 lines):")
!docker logs hiremind-ai --tail 10